<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_03_degradation.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

### A correction to how this is framed

An earlier version of this notebook asked for hydrogen production against
lifetime as the temperature varies, at fixed current density. **That question
has a trivial answer**, and it is worth seeing why before going further.

Hydrogen production follows Faraday's law,

$$
\dot n_{\mathrm{H_2}} = \frac{I\,A}{2F}
$$

so at fixed current density the production rate is **fixed**, whatever the
temperature does. Raising the temperature lowers the resistance and therefore
the voltage, so it reduces the *energy consumed per unit of hydrogen*. It does
not produce more hydrogen.

The questions worth asking are therefore:

- **energy consumption against degradation** at fixed current, where temperature
  genuinely trades one against the other; or
- a **current density sweep**, in which production does change, and degradation
  changes with it.

A second point of the same kind. The degradation law goes as $|i|^n$, so
sweeping the exponent at $i = 1$ A/cm² changes nothing at all: $1^n = 1$ for
every $n$. Sweep the exponent at a current density away from unity, or the
result will be identically flat.


# Ex_10.2 · Notebook 03 — degradation and the trade-off

**Paired with L10.2 · Solid oxide cells**

Produce the trade-off curve as a **result** rather than accepting it from a
slide: electricity per unit of hydrogen against expected lifetime, as
temperature varies at a fixed current density.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex10.2-solid-oxide-cell/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## TODO — sweep temperature and plot both quantities

In [ ]:
ASR_LIMIT = 0.50          # end-of-life threshold, ohm cm2
I_OP = 1.0                # A/cm2

rows = []
for T_c in (650, 700, 750, 800, 850):
    par = pb.SOCParams("SOEC", T_c + 273.15, 0.10, 0.90)
    rate = pb.degradation_rate(I_OP, par)
    hours_to_eol = (ASR_LIMIT - par.ASR_0) / max(rate, 1e-12)
    V = pb.cell_voltage(I_OP, par)
    h2 = pb.hydrogen_rate(I_OP, par) * 3600
    rows.append((T_c, V, h2, rate, hours_to_eol))
    print(f"  {T_c} C:  V {V:.3f} V   H2 {h2:.3e} mol/h   "
          f"deg {rate:.2e}   life {hours_to_eol:,.0f} h")

# TODO: plot cell voltage (electricity per unit of hydrogen) and lifetime
#       against temperature on twin axes, and print total hydrogen over life.
#
#   rows is (T_c, V, h2, rate, hours_to_eol) per temperature. Total hydrogen
#   over life is h2 * hours_to_eol.
#
#   error_table() from course_core formats the sweep as Markdown you can paste
#   straight into the report:
#
#       print(error_table([[f"{T:.0f}", f"{h:.3e}", f"{L:,.0f}", f"{h*L:.3e}"]
#                          for T, _, h, _, L in rows],
#                         ["T [degC]", "H2 [mol/h]", "life [h]", "total [mol]"]))

raise NotImplementedError

**What this model gives.** Total hydrogen over the device's life is production
rate multiplied by lifetime. At a fixed 1 A/cm² the production rate is the same
at every temperature and the lifetime falls as temperature rises, so in this
model the total is largest at the **coolest** setting, about ten times more at
650 °C than at 850 °C. There is no interior optimum in total hydrogen here.

What the cool cell pays is electricity. The cell voltage is the electricity per
unit of hydrogen ($2FV$ joules per mole), and in your table it runs from about
2.07 V at 650 °C down to 1.11 V at 850 °C. So the real trade is electricity per
unit of hydrogen against lifetime. Choosing a point on it needs a price for
electricity and a value for hydrogen, which is what notebook 04 adds.

**How much rests on the exponent.** `deg_exponent` is ESTIMATED. At 1 A/cm²
changing it changes nothing, because $1^n = 1$. Set `I_OP = 0.5` and try 1.0,
1.5 and 2.0: every lifetime moves by the same factor, $0.5^{-(n-1.5)}$, so the
lifetimes change but the ranking of the temperatures does not. At fixed current
the exponent decides *how long*, not *which temperature*.

---

## 1 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. The degradation law is Arrhenius in temperature and a power law in current density. Using the table you printed, say how the degradation rate and the hours to the ASR limit change from 650 to 850 °C. At fixed current, why does a rising ASR mean more electricity for the same hydrogen as the cell ages?
   *→ L10.2 Q6*
2. At a fixed 1 A/cm² the hydrogen column of your table does not change with temperature, because Faraday's law fixes it. Which temperature gives the most hydrogen over life in your table, and why is that not an interior optimum? What does the cool cell pay instead, and which of the three losses is responsible?
   *→ L10.2 Q1, Q6*
3. Change `deg_exponent` to 1.0 and 2.0 at 1 A/cm² and say why nothing moves. Repeat at 0.5 A/cm²: by what factor does each lifetime move, and does the ranking of the temperatures change? The exponent is marked ESTIMATED: what does that say about any optimum that rests on it?
   *→ L10.2 Q6*
4. Sweep current density at one temperature instead. Production rises with current and degradation rises faster, as $|i|^{1.5}$. How does total hydrogen over life scale with current in this model, and where does it peak? What must enter the objective before a higher current is worth running, and how would you report the result when published degradation rates span an order of magnitude?
   *→ L10.2 Q6*


*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.

---

Continue with **[`Ex10.2_04_optimisation.ipynb`](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex10.2-solid-oxide-cell/Ex10.2_04_optimisation.ipynb)**.
